# Adaptive MACD + Bollinger Strategy (BTC & ETH)

Strategy definitions:
- `MACD`: a trend-following long/flat rule based on the MACD line crossing its signal line
- `Bollinger Bands`: a symmetric mean-reversion rule that buys below the lower band, sells above the upper band, and exits at the rolling mean
- `Adaptive switching`: uses MACD in `TREND` regimes and Bollinger in `MEAN_REVERSION` regimes

Audit goals:
- verify all required columns
- remove hidden lookahead from PnL construction
- make the MACD and Bollinger implementations consistent with the original notebooks
- document formulas, assumptions, and economic interpretation clearly


In [1]:
import numpy as np
import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 140)


## 1. Configuration

Data sources:
- the adaptive notebook uses the exported MACD post-trade tables for BTC and ETH
- the original Bollinger baseline is rebuilt from `BTC_full_data.csv` and `ETH_full_data.csv`

Time split:
- Training period: `2018-01-01` to `2024-12-31`
- Test period: `2025-01-01` to `2025-12-31`

Parameter choices kept for source consistency:
- MACD is searched over the same candidate set already used in the notebook
- Bollinger is fixed to the original standalone implementation parameters: `window = 20`, `num_std = 2`
- Trend strength uses the exact requested lookback `L = 30`


In [2]:
MACD_PARAM_GRID = [(12, 26, 9), (16, 20, 15), (18, 50, 13)]
BOLLINGER_WINDOW_GRID = [20]
BOLLINGER_STD_GRID = [2.0]

TREND_LOOKBACK = 30
PRIMARY_TREND_QUANTILE = 0.4
FALLBACK_TREND_QUANTILE = 0.3

TRAIN_START = "2018-01-01"
TRAIN_END = "2024-12-31"
TEST_START = "2025-01-01"
TEST_END = "2025-12-31"

DATA_FILES = {
    "BTC": "btc_macd_16_20_15_post_trade_table.csv",
    "ETH": "eth_macd_18_50_13_post_trade_table.csv",
}

FULL_DATA_FILES = {
    "BTC": "BTC_full_data.csv",
    "ETH": "ETH_full_data.csv",
}

EXPECTED_SOURCE_MACD_PARAMS = {
    "BTC": (16, 20, 15),
    "ETH": (18, 50, 13),
}


## 2. MACD Helper Functions

MACD uses three parameters:
- `a`: fast EMA span
- `b`: slow EMA span
- `c`: signal-line EMA span

We keep the MACD trading rule simple here:
- `MACD > signal`  -> hold a long position
- `MACD <= signal` -> stay flat

That makes the trend-following leg directly comparable across the parameter grid.


In [3]:
def add_macd(data, a=12, b=26, c=9, price_col="Close"):
    out = data.copy()

    if a >= b:
        raise ValueError("Need a < b for MACD.")

    out["ema_fast"] = out[price_col].ewm(span=a, adjust=False).mean()
    out["ema_slow"] = out[price_col].ewm(span=b, adjust=False).mean()
    out["macd"] = out["ema_fast"] - out["ema_slow"]
    out["signal"] = out["macd"].ewm(span=c, adjust=False).mean()
    out["hist"] = out["macd"] - out["signal"]

    return out


def make_position(macd, signal, mode="long_flat", buffer=0.0):
    spread = macd - signal

    if mode == "long_flat":
        pos = np.where(spread > buffer, 1, 0)

    elif mode == "long_short":
        pos = np.where(spread > buffer, 1, -1)

    elif mode == "long_short_flat":
        pos = np.where(
            spread > buffer, 1,
            np.where(spread < -buffer, -1, 0)
        )

    else:
        raise ValueError("mode must be 'long_flat', 'long_short', or 'long_short_flat'")

    return pd.Series(pos, index=macd.index)


## 3. Bollinger Band Helper Function

The notebook keeps the standalone Bollinger specification unchanged:

Log return:

$$
r_t = \ln\left(\frac{P_t}{P_{t-1}}\right)
$$

Bollinger bands:

$$
\text{Rolling Mean}_t = \frac{1}{W} \sum_{i=t-W+1}^{t} P_i
$$

$$
	ext{Upper}_t = 	ext{Rolling Mean}_t + k \sigma_t
$$

$$
	ext{Lower}_t = 	ext{Rolling Mean}_t - k \sigma_t
$$

where:
- `W = 20` is the rolling lookback window
- `k = 2` is the number of rolling standard deviations
- `P_t` is the daily close price

Trading rule:
- if `Close < Lower`, enter a long mean-reversion trade
- if `Close > Upper`, enter a short mean-reversion trade
- if already long, exit when `Close >= Rolling Mean`
- if already short, exit when `Close <= Rolling Mean`


In [4]:
def bollinger_price_mean_reversion(
    df,
    date_col="Date",
    price_col="Close",
    return_col="Log_Return",
    window=20,
    num_std=2
):
    df = df.copy()

    # format date
    df[date_col] = pd.to_datetime(df[date_col])
    df = df.sort_values(date_col).reset_index(drop=True)

    # use price level for Bollinger signal
    df["price"] = df[price_col]

    # if Log_Return missing, calculate it
    if return_col not in df.columns:
        df[return_col] = np.log(df["price"] / df["price"].shift(1))

    # rolling stats on price level
    df["rolling_mean"] = df["price"].rolling(window).mean()
    df["rolling_std"] = df["price"].rolling(window).std()

    # bollinger bands
    df["upper_band"] = df["rolling_mean"] + num_std * df["rolling_std"]
    df["lower_band"] = df["rolling_mean"] - num_std * df["rolling_std"]

    # trading rule: symmetric mean reversion
    position = 0
    positions = []

    for i in range(len(df)):
        price = df.loc[i, "price"]
        mean = df.loc[i, "rolling_mean"]
        upper = df.loc[i, "upper_band"]
        lower = df.loc[i, "lower_band"]

        if pd.isna(mean) or pd.isna(upper) or pd.isna(lower):
            position = 0

        elif position == 0:
            if price < lower:
                position = 1      # long
            elif price > upper:
                position = -1     # short

        elif position == 1:
            if price >= mean:
                position = 0

        elif position == -1:
            if price <= mean:
                position = 0

        positions.append(position)

    df["position"] = positions

    # strategy log return using lagged position
    df["strategy_return"] = df["position"].shift(1).fillna(0) * df[return_col].fillna(0)

    return df


## 4. Evaluation Helpers

The evaluation block follows `evaluation.ipynb` and keeps the no-lookahead rule explicit:

$$
R_t = 	ext{position}_{t-1} 	imes r_t
$$

Practical interpretation:
- the signal observed on day `t` cannot earn day-`t` return immediately
- the held position must be lagged before it is multiplied by the realised return
- trade labels are standardised as `BUY`, `SELL`, and `HOLD`

The validation section later prints:
- `position`
- `Log_Return`
- `strategy_return`

so the PnL construction can be checked row by row.


In [5]:
def position_to_trade(position_series):
    """Translate a position series into trade events."""
    return position_series.diff().fillna(position_series).astype(int)


def trade_action_from_trade(trade_series):
    return np.where(trade_series > 0, "BUY", np.where(trade_series < 0, "SELL", "HOLD"))


def ensure_required_columns(df):
    out = df.copy()

    if "Close" not in out.columns and "price" in out.columns:
        out["Close"] = out["price"]

    if "macd_trade_action" not in out.columns and "macd_trade" in out.columns:
        out["macd_trade_action"] = trade_action_from_trade(out["macd_trade"])
    if "bollinger_band_trade_action" not in out.columns and "bollinger_band_trade" in out.columns:
        out["bollinger_band_trade_action"] = trade_action_from_trade(out["bollinger_band_trade"])
    if "trade_action" not in out.columns and "trade" in out.columns:
        out["trade_action"] = trade_action_from_trade(out["trade"])

    required_cols = [
        "macd_trade",
        "macd_trade_action",
        "macd_position",
        "bollinger_band_trade",
        "bollinger_band_trade_action",
        "bollinger_band_position",
        "trade",
        "trade_action",
        "position",
        "trend_strength",
        "regime",
    ]
    missing_cols = [col for col in required_cols if col not in out.columns]
    if missing_cols:
        raise ValueError(f"Missing required columns: {missing_cols}")

    return out


def prepare_post_trade_df(
    post_trade_df,
    date_col="date",
    price_col="price",
    position_col="position",
    trade_col="trade",
    log_return_col="log_return",
):
    out = post_trade_df.copy()

    if date_col in out.columns:
        out = out.sort_values(date_col).reset_index(drop=True)
    else:
        out = out.reset_index(drop=True)

    if price_col not in out.columns:
        raise ValueError(f"'{price_col}' column is required.")
    if position_col not in out.columns:
        raise ValueError(f"'{position_col}' column is required.")

    out[position_col] = out[position_col].fillna(0).astype(int)

    if price_col in out.columns:
        out["asset_ret"] = out[price_col].pct_change().fillna(0.0)
    elif log_return_col in out.columns:
        out["asset_ret"] = np.expm1(out[log_return_col]).fillna(0.0)
    else:
        raise ValueError(f"Need either '{price_col}' or '{log_return_col}'.")

    if trade_col in out.columns:
        out["trade_size"] = out[trade_col].fillna(0).abs()
    else:
        out["trade_size"] = out[position_col].diff().fillna(out[position_col]).abs()

    return out


def evaluate_daily_post_trade_df(
    post_trade_df,
    date_col="date",
    price_col="price",
    position_col="position",
    trade_col="trade",
    log_return_col="log_return",
    fee=0.0,
    rf_annual=0.03,
    trading_days=365,
):
    out = prepare_post_trade_df(
        post_trade_df=post_trade_df,
        date_col=date_col,
        price_col=price_col,
        position_col=position_col,
        trade_col=trade_col,
        log_return_col=log_return_col,
    )

    out["strategy_ret_gross"] = out[position_col] * out["asset_ret"]
    out["strategy_ret_net"] = out["strategy_ret_gross"] - out["trade_size"] * fee
    out["equity_curve"] = (1 + out["strategy_ret_net"]).cumprod()
    out["cumulative_pnl_series"] = out["equity_curve"] - 1
    out["running_max"] = out["equity_curve"].cummax()
    out["drawdown"] = out["equity_curve"] / out["running_max"] - 1

    n = len(out)
    years = n / trading_days if trading_days > 0 else np.nan
    cumulative_pnl = out["cumulative_pnl_series"].iloc[-1]
    average_daily_pnl = out["strategy_ret_net"].mean()
    max_drawdown = out["drawdown"].min()
    annualised_return = out["equity_curve"].iloc[-1] ** (1 / years) - 1 if years > 0 else np.nan

    daily_std_dev = out["strategy_ret_net"].std()
    annualised_volatility = daily_std_dev * np.sqrt(trading_days)
    rf_daily = (1 + rf_annual) ** (1 / trading_days) - 1
    excess_daily = out["strategy_ret_net"] - rf_daily
    sharpe_ratio = (excess_daily.mean() / daily_std_dev) * np.sqrt(trading_days) if daily_std_dev > 0 else np.nan

    exposure_rate = (out[position_col] != 0).mean()
    total_turnover = out["trade_size"].sum()

    daily_summary = {
        "cumulative_pnl": cumulative_pnl,
        "average_daily_pnl": average_daily_pnl,
        "max_drawdown": max_drawdown,
        "annualised_return": annualised_return,
        "sharpe_ratio_rf_3pct": sharpe_ratio,
        "daily_std_dev": daily_std_dev,
        "annualised_volatility": annualised_volatility,
        "exposure_rate": exposure_rate,
        "total_turnover": total_turnover,
    }

    return out, daily_summary


## 5. Load The Input Data

Each MACD CSV is treated as a source-of-truth reference table:
- `price` and `log_return` provide the close-price series used throughout the notebook
- the original exported MACD `trade`, `trade_action`, and `position` columns are preserved with a `source_macd_` prefix

This allows the notebook to rebuild MACD internally and then verify that the reconstructed post-trade series matches the saved source tables.


In [6]:
def load_strategy_input_data(csv_path):
    df = pd.read_csv(csv_path)
    df["date"] = pd.to_datetime(df["date"])
    df = df.sort_values("date").set_index("date")
    df = df.rename(columns={
        "log_return": "Log_Return",
        "trade": "source_macd_trade",
        "trade_action": "source_macd_trade_action",
        "position": "source_macd_position",
    })

    required_cols = {
        "price",
        "Log_Return",
        "source_macd_trade",
        "source_macd_trade_action",
        "source_macd_position",
    }
    missing_cols = required_cols.difference(df.columns)
    if missing_cols:
        raise ValueError(f"Missing required columns in {csv_path}: {sorted(missing_cols)}")

    df["source_macd_trade"] = df["source_macd_trade"].fillna(0).astype(int)
    df["source_macd_position"] = df["source_macd_position"].fillna(0).astype(int)
    df["source_macd_trade_action"] = df["source_macd_trade_action"].astype(str).str.upper()
    df["Close"] = df["price"]
    return df


In [7]:
btc_df = load_strategy_input_data(DATA_FILES["BTC"])
eth_df = load_strategy_input_data(DATA_FILES["ETH"])


In [8]:
preview_cols = [
    "price",
    "Log_Return",
    "source_macd_trade",
    "source_macd_trade_action",
    "source_macd_position",
]

print("BTC input preview")
display(btc_df[preview_cols].head(3))

print("ETH input preview")
display(eth_df[preview_cols].head(3))


BTC input preview


,price,Log_Return,source_macd_trade,source_macd_trade_action,source_macd_position
date,,,,,
2018-01-01,13657.200195,0.000000,0,HOLD,0
2018-01-02,14982.099609,0.092589,0,HOLD,0
2018-01-03,15201.000000,0.014505,1,BUY,1


ETH input preview


,price,Log_Return,source_macd_trade,source_macd_trade_action,source_macd_position
date,,,,,
2018-01-01,772.640991,0.000000,0,HOLD,0
2018-01-02,884.443970,0.135145,0,HOLD,0
2018-01-03,962.719971,0.084803,1,BUY,1


## 6. Strategy Builder Functions

Implementation rules used below:
- `MACD` is rebuilt from the close-price series and then checked against the exported source MACD tables
- `Bollinger` uses the standalone notebook logic exactly, including mean exits
- base indicators are built on a history-extended dataframe before slicing the 2025 test window, so rolling statistics and EMAs are not reset at `2025-01-01`


In [9]:
def build_macd_signals(df, a, b, c):
    temp = add_macd(df.reset_index().copy(), a=a, b=b, c=c, price_col="Close")

    out = df.copy()
    out["Close"] = out["price"]
    out["ema_fast"] = temp["ema_fast"].values
    out["ema_slow"] = temp["ema_slow"].values
    out["macd_line"] = temp["macd"].values
    out["macd_signal"] = temp["signal"].values
    out["macd_hist"] = temp["hist"].values

    raw_position = make_position(temp["macd"], temp["signal"], mode="long_flat", buffer=0.0).astype(int)
    out["macd_position"] = raw_position.values
    out["macd_trade"] = position_to_trade(out["macd_position"])
    out["macd_trade_action"] = trade_action_from_trade(out["macd_trade"])

    # This is the post-trade series used in the exported MACD CSVs.
    out["macd_post_trade_position"] = raw_position.shift(1).fillna(0).astype(int).values
    out["macd_post_trade_trade"] = position_to_trade(out["macd_post_trade_position"])
    out["macd_post_trade_trade_action"] = trade_action_from_trade(out["macd_post_trade_trade"])
    out["macd_strategy_return"] = out["macd_position"].shift(1).fillna(0) * out["Log_Return"].fillna(0)
    return out


In [10]:
def build_bollinger_signals(df, window, num_std):
    out = df.copy()
    out["Close"] = out["price"]

    bb_input = out.reset_index()[["date", "Close", "Log_Return"]].copy()
    bb = bollinger_price_mean_reversion(
        bb_input,
        date_col="date",
        price_col="Close",
        return_col="Log_Return",
        window=window,
        num_std=num_std,
    ).set_index("date")

    out["rolling_mean"] = bb["rolling_mean"]
    out["rolling_std"] = bb["rolling_std"]
    out["upper"] = bb["upper_band"]
    out["lower"] = bb["lower_band"]
    out["upper_band"] = bb["upper_band"]
    out["lower_band"] = bb["lower_band"]
    out["bollinger_band_position"] = bb["position"].astype(int)
    out["bollinger_band_trade"] = np.select(
        [
            out["Close"] < out["lower"],
            out["Close"] > out["upper"],
        ],
        [1, -1],
        default=0,
    ).astype(int)
    out["bollinger_band_trade_action"] = trade_action_from_trade(out["bollinger_band_trade"])
    out["bollinger_strategy_return"] = out["bollinger_band_position"].shift(1).fillna(0) * out["Log_Return"].fillna(0)
    return out


## 7. Trend-Strength Filter

The adaptive switch follows the exact requested signed efficiency-ratio style formula:

$$
	ext{TrendStrength}_t = 
rac{P_t - P_{t-L}}{\sum_{i=t-L+1}^{t} |P_i - P_{i-1}|}
$$

with:
- `L = 30`
- `P_t = Close_t`

Interpretation:
- the numerator measures net directional move over the lookback window
- the denominator measures the total path length travelled over the same window
- values near `+1` or `-1` indicate efficient trending motion
- values near `0` indicate choppy or mean-reverting movement

Economic interpretation:
- BTC and ETH are often trend-dominant because large directional moves can persist through macro, liquidity, and momentum cycles
- pure mean-reversion rules can therefore underperform or become unstable when price keeps moving away from the band instead of snapping back quickly
- the adaptive filter attempts to use MACD during stronger directional phases and Bollinger only when the market is less trend-efficient


In [11]:
def compute_trend_strength(df, lookback=TREND_LOOKBACK):
    out = df.copy()
    out["Close"] = out["Close"] if "Close" in out.columns else out["price"]

    price_diff = out["Close"] - out["Close"].shift(lookback)
    abs_moves = out["Close"].diff().abs().rolling(lookback).sum()
    out["trend_strength"] = price_diff / abs_moves
    return out["trend_strength"]


def build_adaptive_positions(
    df,
    lookback=TREND_LOOKBACK,
    primary_quantile=PRIMARY_TREND_QUANTILE,
    fallback_quantile=FALLBACK_TREND_QUANTILE,
):
    out = df.copy()
    out["Close"] = out["Close"] if "Close" in out.columns else out["price"]

    if "trend_strength" not in out.columns:
        out["trend_strength"] = compute_trend_strength(out, lookback)

    threshold = out["trend_strength"].abs().quantile(primary_quantile)
    out["regime"] = np.where(
        out["trend_strength"].abs() > threshold,
        "TREND",
        "MEAN_REVERSION",
    )
    quantile_used = float(primary_quantile)

    if (out["regime"] == "TREND").mean() < 0.60:
        threshold = out["trend_strength"].abs().quantile(fallback_quantile)
        out["regime"] = np.where(
            out["trend_strength"].abs() > threshold,
            "TREND",
            "MEAN_REVERSION",
        )
        quantile_used = float(fallback_quantile)

    out["raw_trade"] = np.where(
        out["regime"] == "TREND",
        out["macd_trade"],
        out["bollinger_band_trade"],
    )
    out["raw_trade"] = pd.Series(out["raw_trade"], index=out.index).fillna(0).astype(int)
    out["active_strategy"] = np.where(out["regime"] == "TREND", "MACD", "BOLLINGER")

    position = 0
    positions = []
    trades = []

    for t in range(len(out)):
        signal = int(out["raw_trade"].iloc[t])

        if signal == 1 and position != 1:
            trade = 1
        elif signal == -1 and position != -1:
            trade = -1
        else:
            trade = 0

        position += trade
        position = max(min(position, 1), -1)

        trades.append(trade)
        positions.append(position)

    out["trade"] = np.array(trades, dtype=int)
    out["position"] = np.array(positions, dtype=int)
    out["trade_action"] = trade_action_from_trade(out["trade"])
    out["strategy_return"] = out["position"].shift(1).fillna(0) * out["Log_Return"].fillna(0)

    diagnostics = {
        "trend_lookback": int(lookback),
        "threshold_quantile_used": quantile_used,
        "threshold_value": float(threshold),
        "trend_share": float((out["regime"] == "TREND").mean()),
        "mean_reversion_share": float((out["regime"] == "MEAN_REVERSION").mean()),
    }

    return out, diagnostics


## 8. Optimisation Workflow

Workflow:
1. rebuild and evaluate candidate MACD specifications on the training window
2. rebuild the original Bollinger specification on the training window
3. compute trend-strength thresholds on the training window
4. rebuild the final test dataframe with full indicator history
5. validate source consistency, trade distributions, regime shares, and no-lookahead PnL columns
6. export a final summary table to `trend_strength_filter.csv`


In [12]:
def evaluate_signal_positions(df, position_col, already_post_trade=False, trade_col=None):
    eval_input = df.reset_index()[["date", "price", "Log_Return"]].copy()
    eval_input = eval_input.rename(columns={"Log_Return": "log_return"})

    if already_post_trade:
        eval_input["position"] = df[position_col].fillna(0).astype(int).values
        if trade_col is not None and trade_col in df.columns:
            eval_input["trade"] = df[trade_col].fillna(0).astype(int).values
        else:
            eval_input["trade"] = position_to_trade(eval_input["position"])
    else:
        eval_input["position"] = df[position_col].shift(1).fillna(0).astype(int).values
        eval_input["trade"] = position_to_trade(eval_input["position"])

    _, summary = evaluate_daily_post_trade_df(
        post_trade_df=eval_input,
        date_col="date",
        price_col="price",
        position_col="position",
        trade_col="trade",
        log_return_col="log_return",
        fee=0.0,
        rf_annual=0.03,
        trading_days=365,
    )
    return summary


def evaluate_strategy_original_bollinger(strategy_returns, risk_free_rate=0.03, trading_days=365):
    r = strategy_returns.dropna()

    cumulative = np.exp(r.cumsum())
    cumulative_pnl = cumulative.iloc[-1] - 1
    avg_daily_pnl = r.mean()
    daily_vol = r.std()
    annual_return = r.mean() * trading_days
    annual_vol = r.std() * np.sqrt(trading_days)
    sharpe = (annual_return - risk_free_rate) / annual_vol if annual_vol != 0 else np.nan
    peak = cumulative.cummax()
    drawdown = (cumulative - peak) / peak
    max_dd = drawdown.min()

    return {
        "Cumulative PnL": cumulative_pnl,
        "Average Daily PnL": avg_daily_pnl,
        "Daily Volatility": daily_vol,
        "Annualised Return": annual_return,
        "Volatility": annual_vol,
        "Sharpe Ratio": sharpe,
        "Max Drawdown": max_dd,
    }


def build_original_bollinger_baseline(asset_name):
    full_df = pd.read_csv(FULL_DATA_FILES[asset_name])
    baseline = bollinger_price_mean_reversion(
        full_df,
        date_col="Date",
        price_col="Close",
        return_col="Log_Return",
        window=20,
        num_std=2,
    )

    baseline["date"] = pd.to_datetime(baseline["Date"])
    baseline = baseline.sort_values("date").set_index("date")
    baseline["price"] = baseline["Close"]
    baseline["bollinger_band_trade"] = np.select(
        [
            baseline["Close"] < baseline["lower_band"],
            baseline["Close"] > baseline["upper_band"],
        ],
        [1, -1],
        default=0,
    ).astype(int)
    baseline["bollinger_band_trade_action"] = trade_action_from_trade(baseline["bollinger_band_trade"])
    baseline["bollinger_post_trade_position"] = baseline["position"].shift(1).fillna(0).astype(int)
    baseline["bollinger_post_trade_trade"] = position_to_trade(baseline["bollinger_post_trade_position"])

    full_metrics = evaluate_strategy_original_bollinger(baseline["strategy_return"])
    test_slice = baseline.loc[TEST_START:TEST_END].copy()
    test_eval = evaluate_signal_positions(
        test_slice,
        "bollinger_post_trade_position",
        already_post_trade=True,
        trade_col="bollinger_post_trade_trade",
    )
    test_log_eval = evaluate_strategy_original_bollinger(test_slice["strategy_return"])
    trade_counts = test_slice["bollinger_band_trade"].value_counts().reindex([-1, 0, 1], fill_value=0)

    return {
        "full_df": baseline,
        "full_metrics": full_metrics,
        "test_df": test_slice,
        "test_eval": test_eval,
        "test_log_eval": test_log_eval,
        "trade_counts": trade_counts,
        "turnover": int((test_slice["bollinger_band_trade"] != 0).sum()),
    }


def metrics_close(left, right, keys, atol=1e-12):
    return all(np.isclose(left[key], right[key], atol=atol, equal_nan=True) for key in keys)


In [13]:
def optimise_macd(train_df):
    rows = []
    for a, b, c in MACD_PARAM_GRID:
        macd_train = build_macd_signals(train_df, a, b, c)
        summary = evaluate_signal_positions(
            macd_train,
            "macd_post_trade_position",
            already_post_trade=True,
            trade_col="macd_post_trade_trade",
        )
        rows.append({
            "a": a,
            "b": b,
            "c": c,
            "annualised_return": summary["annualised_return"],
            "sharpe_ratio_rf_3pct": summary["sharpe_ratio_rf_3pct"],
            "cumulative_pnl": summary["cumulative_pnl"],
        })

    macd_opt = pd.DataFrame(rows).sort_values(
        ["annualised_return", "sharpe_ratio_rf_3pct"],
        ascending=False,
    ).reset_index(drop=True)
    best_macd = tuple(macd_opt.loc[0, ["a", "b", "c"]].astype(int).tolist())
    return macd_opt, best_macd


In [14]:
def optimise_bollinger(train_df):
    rows = []
    for window in BOLLINGER_WINDOW_GRID:
        for num_std in BOLLINGER_STD_GRID:
            boll_train = build_bollinger_signals(train_df, window, num_std)
            boll_train["bollinger_post_trade_position"] = boll_train["bollinger_band_position"].shift(1).fillna(0).astype(int)
            boll_train["bollinger_post_trade_trade"] = position_to_trade(boll_train["bollinger_post_trade_position"])
            summary = evaluate_signal_positions(
                boll_train,
                "bollinger_post_trade_position",
                already_post_trade=True,
                trade_col="bollinger_post_trade_trade",
            )
            rows.append({
                "window": window,
                "num_std": float(num_std),
                "annualised_return": summary["annualised_return"],
                "sharpe_ratio_rf_3pct": summary["sharpe_ratio_rf_3pct"],
                "cumulative_pnl": summary["cumulative_pnl"],
                "signal_count": int((boll_train["bollinger_band_trade"] != 0).sum()),
            })

    boll_opt = pd.DataFrame(rows).sort_values(
        ["annualised_return", "sharpe_ratio_rf_3pct"],
        ascending=False,
    ).reset_index(drop=True)
    best_boll = (int(boll_opt.loc[0, "window"]), float(boll_opt.loc[0, "num_std"]))
    return boll_opt, best_boll


In [15]:
def optimise_trend_filter(train_df_with_base_signals):
    rows = []
    for quantile in [PRIMARY_TREND_QUANTILE, FALLBACK_TREND_QUANTILE]:
        adaptive_train, diagnostics = build_adaptive_positions(
            train_df_with_base_signals,
            lookback=TREND_LOOKBACK,
            primary_quantile=quantile,
            fallback_quantile=quantile,
        )
        adaptive_train["adaptive_post_trade_position"] = adaptive_train["position"].shift(1).fillna(0).astype(int)
        adaptive_train["adaptive_post_trade_trade"] = position_to_trade(adaptive_train["adaptive_post_trade_position"])
        summary = evaluate_signal_positions(
            adaptive_train,
            "adaptive_post_trade_position",
            already_post_trade=True,
            trade_col="adaptive_post_trade_trade",
        )
        rows.append({
            "trend_lookback": int(TREND_LOOKBACK),
            "trend_quantile": float(quantile),
            "threshold_value": diagnostics["threshold_value"],
            "annualised_return": summary["annualised_return"],
            "sharpe_ratio_rf_3pct": summary["sharpe_ratio_rf_3pct"],
            "cumulative_pnl": summary["cumulative_pnl"],
            "trend_share_train": diagnostics["trend_share"],
            "used_in_final": False,
        })

    trend_opt = pd.DataFrame(rows)
    _, final_diagnostics = build_adaptive_positions(
        train_df_with_base_signals,
        lookback=TREND_LOOKBACK,
        primary_quantile=PRIMARY_TREND_QUANTILE,
        fallback_quantile=FALLBACK_TREND_QUANTILE,
    )
    trend_opt.loc[
        trend_opt["trend_quantile"] == final_diagnostics["threshold_quantile_used"],
        "used_in_final",
    ] = True

    best_trend = (int(TREND_LOOKBACK), float(final_diagnostics["threshold_quantile_used"]))
    return trend_opt.sort_values("trend_quantile", ascending=False).reset_index(drop=True), best_trend


In [16]:
def optimize_parameters_and_build_test(df, asset_name):
    train_df = df.loc[TRAIN_START:TRAIN_END].copy()
    history_df = df.loc[:TEST_END].copy()

    macd_opt, best_macd = optimise_macd(train_df)
    boll_opt, best_boll = optimise_bollinger(train_df)

    train_bundle = build_macd_signals(train_df, *best_macd)
    train_bundle = build_bollinger_signals(train_bundle, *best_boll)
    train_bundle["trend_strength"] = compute_trend_strength(train_bundle, TREND_LOOKBACK)
    trend_opt, best_trend = optimise_trend_filter(train_bundle)

    history_bundle = build_macd_signals(history_df, *best_macd)
    history_bundle = build_bollinger_signals(history_bundle, *best_boll)
    history_bundle["trend_strength"] = compute_trend_strength(history_bundle, TREND_LOOKBACK)
    history_bundle["bollinger_post_trade_position"] = history_bundle["bollinger_band_position"].shift(1).fillna(0).astype(int)
    history_bundle["bollinger_post_trade_trade"] = position_to_trade(history_bundle["bollinger_post_trade_position"])

    test_df = history_bundle.loc[TEST_START:TEST_END].copy()
    test_df, trend_diagnostics = build_adaptive_positions(
        test_df,
        lookback=TREND_LOOKBACK,
        primary_quantile=PRIMARY_TREND_QUANTILE,
        fallback_quantile=FALLBACK_TREND_QUANTILE,
    )
    test_df["adaptive_post_trade_position"] = test_df["position"].shift(1).fillna(0).astype(int)
    test_df["adaptive_post_trade_trade"] = position_to_trade(test_df["adaptive_post_trade_position"])
    test_df = ensure_required_columns(test_df)

    macd_eval = evaluate_signal_positions(
        test_df,
        "macd_post_trade_position",
        already_post_trade=True,
        trade_col="macd_post_trade_trade",
    )
    boll_eval = evaluate_signal_positions(
        test_df,
        "bollinger_post_trade_position",
        already_post_trade=True,
        trade_col="bollinger_post_trade_trade",
    )
    adaptive_eval = evaluate_signal_positions(
        test_df,
        "adaptive_post_trade_position",
        already_post_trade=True,
        trade_col="adaptive_post_trade_trade",
    )

    eval_table = pd.DataFrame({
        "MACD": macd_eval,
        "Bollinger": boll_eval,
        "Adaptive": adaptive_eval,
    })

    metric_order = [
        "cumulative_pnl",
        "average_daily_pnl",
        "max_drawdown",
        "annualised_return",
        "sharpe_ratio_rf_3pct",
        "daily_std_dev",
        "annualised_volatility",
        "exposure_rate",
        "total_turnover",
    ]
    eval_table = eval_table.loc[metric_order]

    regime_summary = test_df["regime"].value_counts().rename("count").to_frame()
    regime_summary["share"] = regime_summary["count"] / regime_summary["count"].sum()

    source_macd_params = EXPECTED_SOURCE_MACD_PARAMS[asset_name]
    macd_source_position_mismatches = int(
        (history_bundle["macd_post_trade_position"] != history_bundle["source_macd_position"]).sum()
    )
    macd_source_trade_mismatches = int(
        (history_bundle["macd_post_trade_trade"] != history_bundle["source_macd_trade"]).sum()
    )
    macd_source_action_mismatches = int(
        (history_bundle["macd_post_trade_trade_action"] != history_bundle["source_macd_trade_action"]).sum()
    )

    original_bollinger = build_original_bollinger_baseline(asset_name)
    current_trade_counts = test_df["bollinger_band_trade"].value_counts().reindex([-1, 0, 1], fill_value=0)

    signal_distribution_match = current_trade_counts.equals(original_bollinger["trade_counts"])
    turnover_match = int((test_df["bollinger_band_trade"] != 0).sum()) == original_bollinger["turnover"]
    evaluation_match = metrics_close(
        boll_eval,
        original_bollinger["test_eval"],
        ["cumulative_pnl", "annualised_return", "sharpe_ratio_rf_3pct", "total_turnover"],
    )

    bollinger_consistency_table = pd.DataFrame([
        {
            "implementation": "current_rebuild_2025",
            "annualised_return": boll_eval["annualised_return"],
            "sharpe_ratio_rf_3pct": boll_eval["sharpe_ratio_rf_3pct"],
            "cumulative_pnl": boll_eval["cumulative_pnl"],
            "total_turnover": boll_eval["total_turnover"],
            "signal_count": int((test_df["bollinger_band_trade"] != 0).sum()),
            "sell_signals": int(current_trade_counts.loc[-1]),
            "hold_days": int(current_trade_counts.loc[0]),
            "buy_signals": int(current_trade_counts.loc[1]),
        },
        {
            "implementation": "original_bollinger_ipynb_2025",
            "annualised_return": original_bollinger["test_eval"]["annualised_return"],
            "sharpe_ratio_rf_3pct": original_bollinger["test_eval"]["sharpe_ratio_rf_3pct"],
            "cumulative_pnl": original_bollinger["test_eval"]["cumulative_pnl"],
            "total_turnover": original_bollinger["test_eval"]["total_turnover"],
            "signal_count": int(original_bollinger["turnover"]),
            "sell_signals": int(original_bollinger["trade_counts"].loc[-1]),
            "hold_days": int(original_bollinger["trade_counts"].loc[0]),
            "buy_signals": int(original_bollinger["trade_counts"].loc[1]),
        },
    ]).set_index("implementation")

    original_bollinger_full_table = pd.DataFrame([
        {
            "sample": "full_sample_original_log_returns",
            "cumulative_pnl": original_bollinger["full_metrics"]["Cumulative PnL"],
            "annualised_return": original_bollinger["full_metrics"]["Annualised Return"],
            "sharpe_ratio": original_bollinger["full_metrics"]["Sharpe Ratio"],
            "max_drawdown": original_bollinger["full_metrics"]["Max Drawdown"],
        },
        {
            "sample": "test_2025_original_log_returns",
            "cumulative_pnl": original_bollinger["test_log_eval"]["Cumulative PnL"],
            "annualised_return": original_bollinger["test_log_eval"]["Annualised Return"],
            "sharpe_ratio": original_bollinger["test_log_eval"]["Sharpe Ratio"],
            "max_drawdown": original_bollinger["test_log_eval"]["Max Drawdown"],
        },
    ]).set_index("sample")

    return {
        "asset": asset_name,
        "train_df": train_df,
        "test_df": test_df,
        "macd_optimization": macd_opt,
        "bollinger_optimization": boll_opt,
        "trend_filter_optimization": trend_opt,
        "best_macd": best_macd,
        "best_bollinger": best_boll,
        "best_trend_filter": best_trend,
        "trend_diagnostics": trend_diagnostics,
        "trend_strength_describe": test_df["trend_strength"].describe(),
        "bollinger_trade_counts": current_trade_counts,
        "evaluation": eval_table,
        "regime_summary": regime_summary,
        "source_macd_params": source_macd_params,
        "macd_source_position_mismatches": macd_source_position_mismatches,
        "macd_source_trade_mismatches": macd_source_trade_mismatches,
        "macd_source_action_mismatches": macd_source_action_mismatches,
        "bollinger_consistency": {
            "signal_distribution_match": signal_distribution_match,
            "turnover_match": turnover_match,
            "evaluation_match": evaluation_match,
        },
        "bollinger_consistency_table": bollinger_consistency_table,
        "original_bollinger_full_table": original_bollinger_full_table,
        "bollinger_outperforms_macd_test": bool(
            boll_eval["annualised_return"] > macd_eval["annualised_return"]
        ),
        "bollinger_warning_required": bool(
            (boll_eval["annualised_return"] > 0) or (boll_eval["annualised_return"] > macd_eval["annualised_return"])
        ),
    }


## 9. Run The Optimisation And Build The 2025 Test Set


In [17]:
btc_result = optimize_parameters_and_build_test(btc_df, "BTC")
eth_result = optimize_parameters_and_build_test(eth_df, "ETH")


In [18]:
for label, result in [("BTC", btc_result), ("ETH", eth_result)]:
    print(f"{label} best MACD params:", result["best_macd"])
    print(f"{label} source MACD params:", result["source_macd_params"])
    print(f"{label} MACD source position mismatches:", result["macd_source_position_mismatches"])
    print(f"{label} MACD source trade mismatches:", result["macd_source_trade_mismatches"])
    print(f"{label} MACD source trade_action mismatches:", result["macd_source_action_mismatches"])
    print(f"{label} best Bollinger params:", result["best_bollinger"])
    print(f"{label} trend filter settings (lookback, quantile used):", result["best_trend_filter"])
    print(f"{label} threshold value:", round(result["trend_diagnostics"]["threshold_value"], 6))
    print()


BTC best MACD params: (16, 20, 15)
BTC source MACD params: (16, 20, 15)
BTC MACD source position mismatches: 0
BTC MACD source trade mismatches: 0
BTC MACD source trade_action mismatches: 0
BTC best Bollinger params: (20, 2.0)
BTC trend filter settings (lookback, quantile used): (30, 0.3)
BTC threshold value: 0.082302

ETH best MACD params: (18, 50, 13)
ETH source MACD params: (18, 50, 13)
ETH MACD source position mismatches: 0
ETH MACD source trade mismatches: 0
ETH MACD source trade_action mismatches: 0
ETH best Bollinger params: (20, 2.0)
ETH trend filter settings (lookback, quantile used): (30, 0.3)
ETH threshold value: 0.105292



## 10. Final Validation Checks

This section prints the mandatory audit diagnostics:
- Bollinger signal counts and turnover
- `trend_strength.describe()`
- regime shares
- adaptive position counts
- repeated-buy and repeated-sell checks
- the first 20 rows of `position`, `Log_Return`, and `strategy_return`
- the first 30 rows of `trend_strength`, `regime`, `macd_position`, `bollinger_band_position`, and adaptive `position`


In [19]:
validation_cols = [
    "trend_strength",
    "regime",
    "macd_position",
    "bollinger_band_position",
    "position",
]

for label, result in [("BTC", btc_result), ("ETH", eth_result)]:
    df = result["test_df"].copy()
    prev_position = df["position"].shift(1).fillna(0)
    repeated_buys = int(((df["trade"] == 1) & (prev_position == 1)).sum())
    repeated_sells = int(((df["trade"] == -1) & (prev_position == -1)).sum())

    print(f"{label} bollinger_band_trade value counts")
    print(df["bollinger_band_trade"].value_counts())
    print("Turnover:", int((df["bollinger_band_trade"] != 0).sum()))
    print()

    print(f"{label} trend_strength describe()")
    print(df["trend_strength"].describe())
    print()

    print(f"{label} regime distribution")
    print(df["regime"].value_counts(normalize=True))
    print()

    print(f"{label} adaptive position counts")
    print(df["position"].value_counts())
    print("Repeated BUY count:", repeated_buys)
    print("Repeated SELL count:", repeated_sells)
    print("Unique adaptive positions:", sorted(df["position"].dropna().unique().tolist()))
    print()

    display(df[["position", "Log_Return", "strategy_return"]].head(20))
    display(df[validation_cols].head(30))


BTC bollinger_band_trade value counts
bollinger_band_trade
 0    328
 1     19
-1     17
Name: count, dtype: int64
Turnover: 36

BTC trend_strength describe()
count    364.000000
mean       0.009771
std        0.220511
min       -0.458057
25%       -0.150344
50%       -0.025181
75%        0.140888
max        0.624962
Name: trend_strength, dtype: float64

BTC regime distribution
regime
TREND             0.700549
MEAN_REVERSION    0.299451
Name: proportion, dtype: float64

BTC adaptive position counts
position
 0    176
-1    168
 1     20
Name: count, dtype: int64
Repeated BUY count: 0
Repeated SELL count: 0
Unique adaptive positions: [-1, 0, 1]



,position,Log_Return,strategy_return
date,,,
2025-01-01,0,0.010546,0.000000
2025-01-02,0,0.025794,0.000000
2025-01-03,0,0.012519,0.000000
2025-01-04,0,0.001312,0.000000
2025-01-05,0,0.000801,0.000000
2025-01-06,-1,0.037562,0.000000
2025-01-07,-1,-0.051824,0.051824
2025-01-08,-1,-0.019579,0.019579
2025-01-09,-1,-0.027299,0.027299


,trend_strength,regime,macd_position,bollinger_band_position,position
date,,,,,
2025-01-01,-0.027168,MEAN_REVERSION,0,0,0
2025-01-02,0.015930,MEAN_REVERSION,0,0,0
2025-01-03,-0.012245,MEAN_REVERSION,0,0,0
2025-01-04,0.031623,MEAN_REVERSION,0,0,0
2025-01-05,-0.032974,MEAN_REVERSION,0,0,0
2025-01-06,0.041076,MEAN_REVERSION,1,-1,-1
2025-01-07,-0.076613,MEAN_REVERSION,0,-1,-1
2025-01-08,-0.043938,MEAN_REVERSION,0,0,-1
2025-01-09,-0.074609,MEAN_REVERSION,0,0,-1


ETH bollinger_band_trade value counts
bollinger_band_trade
 0    320
 1     23
-1     21
Name: count, dtype: int64
Turnover: 44

ETH trend_strength describe()
count    364.000000
mean      -0.007192
std        0.274118
min       -0.439121
25%       -0.228792
50%       -0.068417
75%        0.119006
max        0.740179
Name: trend_strength, dtype: float64

ETH regime distribution
regime
TREND             0.700549
MEAN_REVERSION    0.299451
Name: proportion, dtype: float64

ETH adaptive position counts
position
 0    150
-1    144
 1     70
Name: count, dtype: int64
Repeated BUY count: 0
Repeated SELL count: 0
Unique adaptive positions: [-1, 0, 1]



,position,Log_Return,strategy_return
date,,,
2025-01-01,0,0.006274,0.0
2025-01-02,0,0.028772,0.0
2025-01-03,0,0.043547,0.0
2025-01-04,0,0.014512,0.0
2025-01-05,0,-0.006474,-0.0
2025-01-06,0,0.014888,0.0
2025-01-07,0,-0.086908,-0.0
2025-01-08,0,-0.016473,-0.0
2025-01-09,0,-0.032665,-0.0


,trend_strength,regime,macd_position,bollinger_band_position,position
date,,,,,
2025-01-01,-0.109334,TREND,0,1,0
2025-01-02,-0.061949,MEAN_REVERSION,0,1,0
2025-01-03,-0.088637,MEAN_REVERSION,0,0,0
2025-01-04,-0.057020,MEAN_REVERSION,0,0,0
2025-01-05,-0.147658,TREND,0,0,0
2025-01-06,-0.122270,TREND,0,0,0
2025-01-07,-0.217256,TREND,0,0,0
2025-01-08,-0.148565,TREND,0,0,0
2025-01-09,-0.154978,TREND,0,0,0


## 11. Evaluation Tables

Strategy formulas used in the final comparison:

MACD:

$$
	ext{MACD}_t = 	ext{EMA}_{fast,t} - 	ext{EMA}_{slow,t}
$$

$$
	ext{Signal}_t = 	ext{EMA}(	ext{MACD}_t)
$$

Adaptive decision rule:
- use MACD when `regime = TREND`
- use Bollinger when `regime = MEAN_REVERSION`

Critical discussion:
- BTC and ETH are usually more trend-dominant than classic mean-reverting assets
- this is why MACD is expected to be more stable than Bollinger over long samples
- the original standalone Bollinger notebook still provides the baseline reference and remains the main consistency check for the mean-reversion leg


In [20]:
warning_text = (
    "This result contradicts the baseline Bollinger strategy performance observed in the original implementation, "
    "where mean-reversion yields negative returns. This discrepancy indicates potential implementation inconsistencies "
    "or unintended biases, requiring further debugging and validation."
)

for label, result in [("BTC", btc_result), ("ETH", eth_result)]:
    print(f"{label} evaluation (Test 2025)")
    display(result["evaluation"].round(6))
    print()

    print(f"{label} Bollinger rebuild vs original bollinger.ipynb (2025 slice)")
    display(result["bollinger_consistency_table"].round(6))
    print("Signal distribution match:", result["bollinger_consistency"]["signal_distribution_match"])
    print("Turnover match:", result["bollinger_consistency"]["turnover_match"])
    print("Evaluation match:", result["bollinger_consistency"]["evaluation_match"])
    print()

    print(f"{label} original Bollinger baseline metrics")
    display(result["original_bollinger_full_table"].round(6))
    print()

    if result["bollinger_warning_required"]:
        print(warning_text)
        print()


BTC evaluation (Test 2025)


,MACD,Bollinger,Adaptive
cumulative_pnl,0.021187,0.320445,-0.024615
average_daily_pnl,0.000154,0.000880,0.000029
max_drawdown,-0.236929,-0.201405,-0.258518
annualised_return,0.021245,0.321454,-0.024681
sharpe_ratio_rf_3pct,0.100144,0.996796,-0.070634
daily_std_dev,0.013893,0.015307,0.013995
annualised_volatility,0.265430,0.292438,0.267367
exposure_rate,0.480769,0.535714,0.516484
total_turnover,21.000000,31.000000,12.000000



BTC Bollinger rebuild vs original bollinger.ipynb (2025 slice)


,annualised_return,sharpe_ratio_rf_3pct,cumulative_pnl,total_turnover,signal_count,sell_signals,hold_days,buy_signals
implementation,,,,,,,,
current_rebuild_2025,0.321454,0.996796,0.320445,31,36,17,328,19
original_bollinger_ipynb_2025,0.321454,0.996796,0.320445,31,36,17,328,19


Signal distribution match: True
Turnover match: True
Evaluation match: True

BTC original Bollinger baseline metrics


...

,MACD,Bollinger,Adaptive
cumulative_pnl,0.109023,-0.041460,-0.238826
average_daily_pnl,0.000611,0.000343,-0.000350
max_drawdown,-0.379752,-0.384420,-0.564782
annualised_return,0.109339,-0.041571,-0.239396
sharpe_ratio_rf_3pct,0.391857,0.165781,-0.289475
daily_std_dev,0.025820,0.030239,0.028412
annualised_volatility,0.493294,0.577713,0.542820
exposure_rate,0.403846,0.546703,0.585165
total_turnover,15.000000,31.000000,15.000000



ETH Bollinger rebuild vs original bollinger.ipynb (2025 slice)


,annualised_return,sharpe_ratio_rf_3pct,cumulative_pnl,total_turnover,signal_count,sell_signals,hold_days,buy_signals
implementation,,,,,,,,
current_rebuild_2025,-0.041571,0.165781,-0.04146,31,44,21,320,23
original_bollinger_ipynb_2025,-0.041571,0.165781,-0.04146,31,44,21,320,23


Signal distribution match: True
Turnover match: True
Evaluation match: True

ETH original Bollinger baseline metrics


,cumulative_pnl,annualised_return,sharpe_ratio,max_drawdown
sample,,,,
full_sample_original_log_returns,-0.949464,-0.373006,-0.610087,-0.970129
test_2025_original_log_returns,0.102851,0.098167,0.118583,-0.340767


## 12. Optimisation, Diagnostics, And Export

Final diagnostics reported below:
- train-sample optimisation tables
- regime-filter diagnostics
- a final cross-asset summary table for presentation
- export of `final_table` to `trend_strength_filter.csv`

The summary table is designed to be readable for:
- professors reviewing the methodology
- business stakeholders who need the headline outcomes
- technical readers who need the implementation checks


In [21]:
print("MACD optimisation results (BTC)")
display(btc_result["macd_optimization"].round(6))
print("Bollinger optimisation results (BTC)")
display(btc_result["bollinger_optimization"].round(6))
print("Trend-filter optimisation results (BTC)")
display(btc_result["trend_filter_optimization"].round(6))
print()
print("MACD optimisation results (ETH)")
display(eth_result["macd_optimization"].round(6))
print("Bollinger optimisation results (ETH)")
display(eth_result["bollinger_optimization"].round(6))
print("Trend-filter optimisation results (ETH)")
display(eth_result["trend_filter_optimization"].round(6))

final_rows = []
for label, result in [("BTC", btc_result), ("ETH", eth_result)]:
    evaluation = result["evaluation"]
    trend_diag = result["trend_diagnostics"]
    full_baseline = result["original_bollinger_full_table"].loc["full_sample_original_log_returns"]

    final_rows.append({
        "asset": label,
        "macd_params": str(result["best_macd"]),
        "source_macd_params": str(result["source_macd_params"]),
        "bollinger_params": str(result["best_bollinger"]),
        "trend_lookback": int(result["best_trend_filter"][0]),
        "trend_quantile_used": float(result["best_trend_filter"][1]),
        "trend_threshold": float(trend_diag["threshold_value"]),
        "trend_share": float(trend_diag["trend_share"]),
        "mean_reversion_share": float(trend_diag["mean_reversion_share"]),
        "macd_annualised_return_2025": float(evaluation.loc["annualised_return", "MACD"]),
        "bollinger_annualised_return_2025": float(evaluation.loc["annualised_return", "Bollinger"]),
        "adaptive_annualised_return_2025": float(evaluation.loc["annualised_return", "Adaptive"]),
        "bollinger_full_sample_annualised_return": float(full_baseline["annualised_return"]),
        "bollinger_full_sample_sharpe": float(full_baseline["sharpe_ratio"]),
        "macd_source_position_mismatches": int(result["macd_source_position_mismatches"]),
        "macd_source_trade_mismatches": int(result["macd_source_trade_mismatches"]),
        "bollinger_signal_distribution_match": bool(result["bollinger_consistency"]["signal_distribution_match"]),
        "bollinger_turnover_match": bool(result["bollinger_consistency"]["turnover_match"]),
        "bollinger_evaluation_match": bool(result["bollinger_consistency"]["evaluation_match"]),
        "bollinger_outperforms_macd_test": bool(result["bollinger_outperforms_macd_test"]),
    })

final_table = pd.DataFrame(final_rows)
display(final_table.round(6))
final_table.to_csv("trend_strength_filter.csv", index=False)
print("Saved trend_strength_filter.csv")


MACD optimisation results (BTC)


,a,b,c,annualised_return,sharpe_ratio_rf_3pct,cumulative_pnl
0,16,20,15,0.549450,1.108278,20.492436
1,18,50,13,0.538466,1.095754,19.447534
2,12,26,9,0.466772,0.999405,13.636952


Bollinger optimisation results (BTC)


,window,num_std,annualised_return,sharpe_ratio_rf_3pct,cumulative_pnl,signal_count
0,20,2.0,-0.315622,-0.501208,-0.929827,282


Trend-filter optimisation results (BTC)


...

,window,num_std,annualised_return,sharpe_ratio_rf_3pct,cumulative_pnl,signal_count
0,20,2.0,-0.487257,-0.694499,-0.990717,292


Trend-filter optimisation results (ETH)


,trend_lookback,trend_quantile,threshold_value,annualised_return,sharpe_ratio_rf_3pct,cumulative_pnl,trend_share_train,used_in_final
0,30,0.4,0.147457,-0.273134,-0.117771,-0.892990,0.592882,False
1,30,0.3,0.109513,-0.192975,0.001159,-0.777312,0.691826,True


,asset,macd_params,source_macd_params,bollinger_params,trend_lookback,trend_quantile_used,trend_threshold,trend_share,mean_reversion_share,macd_annualised_return_2025,bollinger_annualised_return_2025,adaptive_annualised_return_2025,bollinger_full_sample_annualised_return,bollinger_full_sample_sharpe,macd_source_position_mismatches,macd_source_trade_mismatches,bollinger_signal_distribution_match,bollinger_turnover_match,bollinger_evaluation_match,bollinger_outperforms_macd_test
0,BTC,"(16, 20, 15)","(16, 20, 15)","(20, 2.0)",30,0.3,0.082302,0.700549,0.299451,0.021245,0.321454,-0.024681,-0.186325,-0.421224,0,0,True,True,True,True
1,ETH,"(18, 50, 13)","(18, 50, 13)","(20, 2.0)",30,0.3,0.105292,0.700549,0.299451,0.109339,-0.041571,-0.239396,-0.373006,-0.610087,0,0,True,True,True,False


Saved trend_strength_filter.csv
